# Lusitano Q3–Q5 External Deployment Validation
Runs the same 10% self-filter + 5k PatchCore configuration under FP32, FP16, and INT8/TensorRT for external validation.


In [ ]:
# ============================================================
# LUSITANO Q3-Q5 — COLAB 2.11 / CUDA 12.8 BOOTSTRAP
# ============================================================
# Current Colab stack (Aug 2026):
#   PyTorch 2.11.0 + CUDA 12.8
#   Torchvision 0.26.0 + CUDA 12.8
# Matching deployment stack:
#   Torch-TensorRT 2.11.0 + CUDA 12.8
#   TensorRT 10.15.1 + CUDA 12
#
# SAFETY:
# - NEVER replace PyTorch or Torchvision.
# - Install Torch-TensorRT with --no-deps so it cannot alter torch.
# - Pin the TensorRT runtime expected by Torch-TensorRT 2.11.
# - Verify native imports in a CHILD PROCESS before the main cell.
# ============================================================

import importlib.metadata as md
import subprocess
import sys


def dist_version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None


versions = {
    "torch": dist_version("torch"),
    "torchvision": dist_version("torchvision"),
    "torch-tensorrt": dist_version("torch-tensorrt"),
    "tensorrt-cu12": dist_version("tensorrt-cu12"),
    "nvidia-modelopt": dist_version("nvidia-modelopt"),
    "dllist": dist_version("dllist"),
}
print("Installed metadata BEFORE bootstrap:", versions)

# Do not mutate the Colab PyTorch foundation.
if versions["torch"] is None or not versions["torch"].startswith("2.11.0"):
    raise RuntimeError(
        f"This notebook expects the current Colab PyTorch 2.11 runtime; found {versions['torch']}. "
        "Use a fresh L4 GPU runtime. PyTorch will NOT be replaced by this notebook."
    )
if versions["torchvision"] is None or not versions["torchvision"].startswith("0.26.0"):
    raise RuntimeError(
        f"Expected torchvision 0.26.x with PyTorch 2.11; found {versions['torchvision']}. "
        "Use a fresh L4 GPU runtime. Torchvision will NOT be replaced."
    )
if "+cu128" not in versions["torch"] or "+cu128" not in versions["torchvision"]:
    raise RuntimeError(
        f"Expected CUDA-12.8 PyTorch/Torchvision wheels; found torch={versions['torch']}, "
        f"torchvision={versions['torchvision']}."
    )

# Install ONLY the matching Torch-TensorRT wheel. --no-deps protects PyTorch.
trt_torch_version = versions["torch-tensorrt"]
if trt_torch_version is None or not trt_torch_version.startswith("2.11.0+cu128"):
    print("\nInstalling matching Torch-TensorRT 2.11.0+cu128 (PyTorch untouched)...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--no-deps",
        "torch-tensorrt==2.11.0+cu128",
        "--index-url", "https://download.pytorch.org/whl/cu128",
    ])
else:
    print("Matching Torch-TensorRT already installed:", trt_torch_version)

# Torch-TensorRT 2.11 release targets TensorRT 10.15.1.
trt_runtime_version = versions["tensorrt-cu12"]
if trt_runtime_version is None or not trt_runtime_version.startswith("10.15.1.29"):
    print("Installing matching TensorRT 10.15.1.29 CUDA-12 runtime...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "tensorrt-cu12==10.15.1.29",
    ])
else:
    print("Matching TensorRT runtime already installed:", trt_runtime_version)

# torch-tensorrt is installed with --no-deps to protect Colab PyTorch,
# so explicitly install its lightweight Python dependency if Colab lacks it.
if dist_version("dllist") is None:
    print("Installing Torch-TensorRT dependency: dllist...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "dllist"
    ])
else:
    print("dllist already installed:", dist_version("dllist"))

# Q5 uses NVIDIA Model Optimizer. Bare nvidia-modelopt includes modelopt.torch requirements.
if versions["nvidia-modelopt"] is None:
    print("Installing NVIDIA ModelOpt for Q5 INT8 PTQ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "nvidia-modelopt",
    ])
else:
    print("NVIDIA ModelOpt already installed:", versions["nvidia-modelopt"])

# Verify native stack outside the notebook kernel first.
probe_code = r'''import warnings
warnings.filterwarnings("ignore")
import torch
# PyTorch 2.11 compatibility reset: torch.export still queries the legacy cuDNN TF32 getter.
# Setting the legacy aggregate flag to False realigns cuDNN conv/RNN state even if an earlier cell
# used the newer fp32_precision API. Do NOT set any new fp32_precision flags in this notebook.
torch.backends.cudnn.allow_tf32 = False
import torchvision
import tensorrt as trt

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda:", torch.version.cuda)
print("TensorRT:", trt.__version__)
logger = trt.Logger(trt.Logger.ERROR)
builder = trt.Builder(logger)
if builder is None:
    raise RuntimeError("TensorRT Builder creation failed")
print("TensorRT Builder: OK")

# IMPORTANT: do NOT mutate PyTorch TF32 global/per-op state here.
# PyTorch 2.11 torch.export internally snapshots cuDNN flags via a legacy getter;
# manually setting the new fp32_precision hierarchy before export can make that
# snapshot path report a mixed-state error. Q3 strict FP32 is instead enforced
# by Torch-TensorRT's compile-time disable_tf32=True option.
# Verify the legacy getter is readable after the compatibility reset.
_cudnn_tf32 = torch.backends.cudnn.allow_tf32
print('cuDNN legacy TF32 getter after reset:', _cudnn_tf32)
import torch_tensorrt
print("torch_tensorrt:", torch_tensorrt.__version__)
print("ModelOpt package metadata:", __import__("importlib.metadata").metadata.version("nvidia-modelopt"))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO_GPU")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not available")

class TinyConv(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Conv2d(3, 8, 3, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(8, 8, 3, padding=1),
            torch.nn.AdaptiveAvgPool2d((1, 1)),
        )
    def forward(self, x):
        return self.net(x).flatten(1)

# Exact Q3-style FP32 compile smoke test.
fp32_model = TinyConv().cuda().eval()
fp32_spec = torch_tensorrt.Input(
    min_shape=(1, 3, 32, 32), opt_shape=(2, 3, 32, 32), max_shape=(4, 3, 32, 32),
    dtype=torch.float32,
)
fp32_trt = torch_tensorrt.compile(
    fp32_model, ir="dynamo", inputs=[fp32_spec], min_block_size=1,
    use_explicit_typing=True, disable_tf32=True,
)
with torch.no_grad():
    y = fp32_trt(torch.zeros(1, 3, 32, 32, device="cuda", dtype=torch.float32))
print("FP32 TensorRT compile+forward: OK", tuple(y.shape), y.dtype)
del fp32_trt, fp32_model, y

# Exact Q4-style FP16 compile smoke test.
fp16_model = TinyConv().half().cuda().eval()
fp16_spec = torch_tensorrt.Input(
    min_shape=(1, 3, 32, 32), opt_shape=(2, 3, 32, 32), max_shape=(4, 3, 32, 32),
    dtype=torch.float16,
)
fp16_trt = torch_tensorrt.compile(
    fp16_model, ir="dynamo", inputs=[fp16_spec], min_block_size=1,
    use_explicit_typing=True, disable_tf32=True,
)
with torch.no_grad():
    y = fp16_trt(torch.zeros(1, 3, 32, 32, device="cuda", dtype=torch.float16))
print("FP16 TensorRT compile+forward: OK", tuple(y.shape), y.dtype)
del fp16_trt, fp16_model, y

# Q5-style ModelOpt INT8 -> torch.export -> TensorRT smoke test.
import modelopt.torch.quantization as mtq
from modelopt.torch.quantization.utils import export_torch_mode
int8_model = TinyConv().cuda().eval()
for p in int8_model.parameters():
    p.requires_grad = False

def calibration_loop(model):
    model.eval()
    with torch.no_grad():
        for _ in range(2):
            model(torch.randn(2, 3, 32, 32, device="cuda", dtype=torch.float32))

mtq.quantize(int8_model, mtq.INT8_DEFAULT_CFG, forward_loop=calibration_loop)
batch_dim = torch.export.Dim("batch", min=1, max=4)
example = torch.randn(2, 3, 32, 32, device="cuda", dtype=torch.float32)
with export_torch_mode():
    ep = torch.export.export(
        int8_model, (example,), dynamic_shapes=({0: batch_dim},), strict=False
    )
int8_spec = torch_tensorrt.Input(
    min_shape=(1, 3, 32, 32), opt_shape=(2, 3, 32, 32), max_shape=(4, 3, 32, 32),
    dtype=torch.float32,
)
int8_trt = torch_tensorrt.dynamo.compile(ep, arg_inputs=[int8_spec], min_block_size=1)
with torch.no_grad():
    y = int8_trt(torch.zeros(1, 3, 32, 32, device="cuda", dtype=torch.float32))
print("INT8 ModelOpt export+TensorRT compile+forward: OK", tuple(y.shape), y.dtype)
torch.cuda.synchronize()
print("CHILD INTEGRATION PREFLIGHT: OK")
'''

probe = subprocess.run(
    [sys.executable, "-c", probe_code],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print("\n" + probe.stdout)
if probe.returncode != 0:
    raise RuntimeError(
        f"Torch-TensorRT/TensorRT child preflight failed (exit code {probe.returncode}). "
        "The main Colab kernel was protected."
    )

print("Installed metadata AFTER bootstrap:", {
    "torch": dist_version("torch"),
    "torchvision": dist_version("torchvision"),
    "torch-tensorrt": dist_version("torch-tensorrt"),
    "tensorrt-cu12": dist_version("tensorrt-cu12"),
    "nvidia-modelopt": dist_version("nvidia-modelopt"),
    "dllist": dist_version("dllist"),
})
print("SAFE PREFLIGHT: OK")
print("FP32/FP16/INT8 TensorRT integration preflight passed in a clean child process.")
print("Run the main Q3-Q5 cell from a CLEAN kernel state.")


In [ ]:
# ============================================================
# IMPORTS + DRIVE
# Stable stack is already aligned by the setup cell above.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

import copy
import gc
import os
import random
import shutil
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
# PyTorch 2.11 compatibility reset for torch.export/Torch-TensorRT.
# This is the documented workaround for the 2.11 legacy cuDNN TF32 getter bug after newer
# fp32_precision APIs have been touched in the same process. We do not use the new TF32 API here.
torch.backends.cudnn.allow_tf32 = False
import torch.nn as nn

# Import torchvision before Torch-TensorRT / ModelOpt.
import torchvision
from torchvision import transforms
from torchvision.models import EfficientNet_B5_Weights, efficientnet_b5

warnings.filterwarnings(
    "ignore",
    message=r"Failed to import modelopt huggingface plugin due to:.*",
)
warnings.filterwarnings(
    "ignore",
    message=r"Failed to import modelopt transformers trainer plugin due to:.*",
)
warnings.filterwarnings(
    "ignore",
    message=r"transformers .* is not tested with current version of modelopt.*",
)

warnings.filterwarnings(
    "ignore",
    message=r"Failed to import modelopt .* plugin due to:.*",
)

import torch_tensorrt
import tensorrt as trt
# ModelOpt is intentionally imported only at Q5, after Q3/Q4 are saved.

from IPython.display import display
from PIL import Image, ImageFile
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

print("PyTorch       :", torch.__version__)
print("PyTorch CUDA  :", torch.version.cuda)
print("Torchvision   :", torchvision.__version__)
print("Torch-TensorRT:", getattr(torch_tensorrt, "__version__", "unknown"))
print("TensorRT      :", trt.__version__)
print("GPU           :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA GPU")

if not torch.cuda.is_available():
    raise RuntimeError("Enable an NVIDIA GPU in Colab: Runtime > Change runtime type > GPU.")

# Fast smoke test before any dataset work.
_smoke = efficientnet_b5(weights=None).to("cuda").eval()
with torch.no_grad():
    _ = _smoke(torch.zeros(1, 3, 224, 224, device="cuda"))
del _smoke
torch.cuda.empty_cache()
print("Environment smoke test: OK")



# C. EXACT EXPERIMENT CONFIGURATION
# ============================================================
SEED = 42
IMG_SIZE = 448
BATCH_SIZE = 16
NUM_WORKERS = 0
FEATURE_LAYER = 7
PATCHES_PER_IMAGE = 200
PRE_POOL = 400_000
STREAM_POOL_MARGIN = 50_000
FILTER_SCORING_BANK_SIZE = 20_000
FINAL_MEMORY_SIZE = 5_000
FILTER_PERCENT = 10.0
CORESET_CHUNK = 40_000
NN_CHUNK = 5_000
THRESH_SAMPLE_IMAGES = 2_000
WARMUP_BATCHES = 5
INT8_CALIBRATION_IMAGES = 512
INT8_CALIBRATION_BATCH = 16
SELECTED_CATEGORIES = ["lusitano"]

# Located in your Drive under:
# <YOUR_DATASET_FOLDER>/Lusitano_Dataset
DRIVE_DATASET_ROOT = Path(
    "/content/drive/MyDrive/<YOUR_DATASET_FOLDER>/"
    "Lusitano_Dataset"
)

# Normalized local MVTec-like structure created by this notebook:
# /content/Lusitano_AD/lusitano/train/good
# /content/Lusitano_AD/lusitano/test/good
# /content/Lusitano_AD/lusitano/test/anomaly
LOCAL_DATASET_ROOT = Path("/content/Lusitano_AD")
RECOPY_LOCAL_DATASET = False
DRIVE_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/<YOUR_OUTPUT_FOLDER>/"
    "Lusitano_Q3_Q5_quantization_validation"
)
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Critical Phase-1 artifacts are checkpointed in Drive. If TensorRT/ModelOpt
# ever fails later, Q3 bank construction does NOT need to be repeated.
LOCAL_ARTIFACT_DIR = DRIVE_OUTPUT_DIR / "artifacts_seed42"
LOCAL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = DRIVE_OUTPUT_DIR / "Lusitano_Q3_Q5_results_seed42.csv"
DELTAS_CSV = DRIVE_OUTPUT_DIR / "Lusitano_Q3_Q5_pairwise_deltas_seed42.csv"
RANKINGS_CSV = DRIVE_OUTPUT_DIR / "Lusitano_Q3_self_filter_rankings_seed42.csv"
RUN_INFO_CSV = DRIVE_OUTPUT_DIR / "Lusitano_Q3_Q5_run_info_seed42.csv"

DEVICE = "cuda"
if not torch.cuda.is_available():
    raise RuntimeError("Enable an NVIDIA GPU in Colab: Runtime > Change runtime type > GPU.")

ImageFile.LOAD_TRUNCATED_IMAGES = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# IMPORTANT: leave PyTorch 2.11 TF32 state at its clean runtime defaults.
# Do not write either the legacy allow_tf32 flags or the new fp32_precision
# hierarchy. torch.export internally snapshots cuDNN state; a manually mixed
# hierarchy can make export fail before TensorRT compilation begins.
# Q3/Q4 TensorRT engines use disable_tf32=True in their compile settings.
print("PyTorch TF32 globals: untouched; TensorRT Q3/Q4 use disable_tf32=True")

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Torch-TensorRT:", getattr(torch_tensorrt, "__version__", "unknown"))
print("TensorRT:", trt.__version__)
print("Lusitano Drive path:", DRIVE_DATASET_ROOT)
print("Results:", DRIVE_OUTPUT_DIR)


# D. HELPERS
# ============================================================
IMG_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp",
}


def bytes_to_mb(value):
    return float(value) / (1024.0 ** 2)


def tensor_size_mb(tensor):
    return bytes_to_mb(tensor.numel() * tensor.element_size())


def floating_parameter_count(model):
    count = 0

    for p in model.parameters():
        if p.is_floating_point():
            count += p.numel()

    for b in model.buffers():
        if b.is_floating_point():
            count += b.numel()

    return int(count)


def estimated_backbone_storage_mb(model, bytes_per_value):
    return bytes_to_mb(
        floating_parameter_count(model) * bytes_per_value
    )


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def reset_peak_gpu_memory():
    clear_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()


def peak_gpu_memory_mb():
    torch.cuda.synchronize()
    return bytes_to_mb(torch.cuda.max_memory_allocated())


def list_images(folder):
    return [
        p for p in sorted(folder.rglob("*"))
        if p.is_file() and p.suffix.lower() in IMG_EXTENSIONS
    ]


def copy_dataset_to_local(drive_root, local_root, recopy=False):
    """Fast/resumable Drive -> /content staging without deleting partial progress."""
    from concurrent.futures import ThreadPoolExecutor, as_completed

    train_src = drive_root / "nondefects" / "nondefects"
    test_good_src = drive_root / "test" / "test" / "non-defects"
    test_anomaly_src = drive_root / "test" / "test" / "defects"

    for required in [train_src, test_good_src, test_anomaly_src]:
        if not required.exists():
            raise FileNotFoundError(f"Required Lusitano folder not found: {required}")

    category_root = local_root / "lusitano"
    train_dst = category_root / "train" / "good"
    test_good_dst = category_root / "test" / "good"
    test_anomaly_dst = category_root / "test" / "anomaly"

    if recopy and local_root.exists():
        print("RECOPY_LOCAL_DATASET=True -> removing old local staging copy")
        shutil.rmtree(local_root)

    mappings = [
        (train_src, train_dst, "train/good"),
        (test_good_src, test_good_dst, "test/good"),
        (test_anomaly_src, test_anomaly_dst, "test/anomaly"),
    ]
    for _, dst, _ in mappings:
        dst.mkdir(parents=True, exist_ok=True)

    def count_images_fast(folder):
        count = 0
        for root, _, files in os.walk(folder):
            count += sum(1 for name in files if Path(name).suffix.lower() in IMG_EXTENSIONS)
        return count

    def fallback_parallel_copy(src, dst, label, workers=8):
        jobs = []
        for root, _, files in os.walk(src):
            root_path = Path(root)
            rel_root = root_path.relative_to(src)
            out_root = dst / rel_root
            out_root.mkdir(parents=True, exist_ok=True)
            for name in files:
                s = root_path / name
                d = out_root / name
                try:
                    if d.exists() and d.stat().st_size == s.stat().st_size:
                        continue
                except OSError:
                    pass
                jobs.append((s, d))

        print(f"{label}: {len(jobs)} files still need copying")
        if not jobs:
            return

        def copy_one(pair):
            s, d = pair
            d.parent.mkdir(parents=True, exist_ok=True)
            tmp = d.with_suffix(d.suffix + ".part")
            shutil.copyfile(s, tmp)
            os.replace(tmp, d)
            return 1

        done = 0
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futures = [pool.submit(copy_one, job) for job in jobs]
            for future in as_completed(futures):
                future.result()
                done += 1
                if done % 100 == 0 or done == len(jobs):
                    print(f"  {label}: {done}/{len(jobs)} copied")

    print("Staging Lusitano on local Colab storage (FAST + RESUMABLE)...")
    start_time = time.perf_counter()

    rsync_bin = shutil.which("rsync")
    for src, dst, label in mappings:
        print(f"\n[{label}]")
        if rsync_bin:
            subprocess.run([
                rsync_bin, "-r", "--size-only", "--partial", "--inplace",
                "--info=progress2", str(src) + "/", str(dst) + "/"
            ], check=True)
        else:
            fallback_parallel_copy(src, dst, label, workers=8)

    elapsed = time.perf_counter() - start_time

    src_counts = {
        "train/good": count_images_fast(train_src),
        "test/good": count_images_fast(test_good_src),
        "test/anomaly": count_images_fast(test_anomaly_src),
    }
    dst_counts = {
        "train/good": count_images_fast(train_dst),
        "test/good": count_images_fast(test_good_dst),
        "test/anomaly": count_images_fast(test_anomaly_dst),
    }

    print("\nLusitano staging verification:")
    for key in src_counts:
        print(f"  {key}: {dst_counts[key]}/{src_counts[key]} images")
        if dst_counts[key] != src_counts[key]:
            raise RuntimeError(
                f"Incomplete local staging for {key}: "
                f"{dst_counts[key]}/{src_counts[key]} images. Rerun this cell to resume."
            )

    print(f"Local staging complete in {elapsed:.1f} sec")
    return local_root, elapsed

# ============================================================
# E. DATASET
# ============================================================
DATASET_ROOT, DATASET_COPY_TIME_SEC = copy_dataset_to_local(
    DRIVE_DATASET_ROOT,
    LOCAL_DATASET_ROOT,
    RECOPY_LOCAL_DATASET,
)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


class ImagePathDataset(Dataset):
    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]

        with Image.open(path) as image:
            image = image.convert("RGB")
            tensor = transform(image)

        return tensor, int(index), str(path)


class TestDataset(Dataset):
    def __init__(self, items):
        self.items = list(items)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        path, label = self.items[index]

        with Image.open(path) as image:
            image = image.convert("RGB")
            tensor = transform(image)

        return tensor, int(label), str(path)


def make_image_loader(paths, batch_size=BATCH_SIZE):
    return DataLoader(
        ImagePathDataset(paths),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
    )


def make_test_loader(items, batch_size=BATCH_SIZE):
    return DataLoader(
        TestDataset(items),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
    )


def discover_categories(root):
    categories = []

    for folder in sorted(root.iterdir()):
        if not folder.is_dir():
            continue

        if folder.name.lower() == "samples":
            continue

        if (
            (folder / "train" / "good").exists()
            and (folder / "test" / "good").exists()
            and (folder / "test" / "anomaly").exists()
        ):
            categories.append(folder.name)

    return categories


all_categories = discover_categories(DATASET_ROOT)

if SELECTED_CATEGORIES is None:
    categories = all_categories
else:
    missing = sorted(set(SELECTED_CATEGORIES) - set(all_categories))

    if missing:
        raise ValueError(
            f"Unknown/invalid Lusitano categories: {missing}"
        )

    categories = list(SELECTED_CATEGORIES)


if not categories:
    raise RuntimeError("No valid Lusitano dataset found.")


print("\nValidation dataset:")
for category in categories:
    print(" -", category)


# ============================================================
# F. DEPLOYMENT-CORRECT EFFICIENTNET-B5 LAYER-7 EXTRACTOR
# ============================================================
class EfficientNetB5Layer7(nn.Module):
    """
    Returns exactly the output of EfficientNet-B5 features[7].

    Unlike the old hook implementation, inference stops after layer 7,
    because later EfficientNet layers are not required by PatchCore.
    """

    def __init__(self):
        super().__init__()

        full_model = efficientnet_b5(
            weights=EfficientNet_B5_Weights.DEFAULT
        )

        self.features = nn.Sequential(
            *list(full_model.features.children())[
                : FEATURE_LAYER + 1
            ]
        )

        for parameter in self.parameters():
            parameter.requires_grad = False

    def forward(self, images):
        feature_map = self.features(images)

        batch, channels, height, width = feature_map.shape

        return (
            feature_map
            .reshape(batch, channels, height * width)
            .permute(0, 2, 1)
            .contiguous()
        )


# Used only for memory-bank construction + self-filtering.
training_extractor = (
    EfficientNetB5Layer7()
    .to(DEVICE)
    .eval()
)

with torch.no_grad():
    dummy = torch.zeros(
        1, 3, IMG_SIZE, IMG_SIZE,
        device=DEVICE,
        dtype=torch.float32,
    )

    dummy_features = training_extractor(dummy)


FEATURE_DIMENSION = int(dummy_features.shape[-1])
PATCH_GRID_COUNT = int(dummy_features.shape[1])

FP32_BACKBONE_EST_MB = estimated_backbone_storage_mb(
    training_extractor, 4
)
FP16_BACKBONE_EST_MB = estimated_backbone_storage_mb(
    training_extractor, 2
)
INT8_BACKBONE_EST_MB = estimated_backbone_storage_mb(
    training_extractor, 1
)

print("\nLayer-7 extractor")
print("Feature dimension:", FEATURE_DIMENSION)
print("Patch-grid count:", PATCH_GRID_COUNT)
print(
    f"Estimated FP32 backbone weight storage: "
    f"{FP32_BACKBONE_EST_MB:.3f} MB"
)
print(
    f"Estimated FP16 backbone weight storage: "
    f"{FP16_BACKBONE_EST_MB:.3f} MB"
)
print(
    f"Estimated INT8 backbone weight storage: "
    f"{INT8_BACKBONE_EST_MB:.3f} MB"
)

del dummy, dummy_features
clear_memory()


# ============================================================
# G. PATCH POOL + GREEDY CORESET
# ============================================================
def compact_candidate_pool(
    feature_chunks,
    image_id_chunks,
    key_chunks,
    maximum_size,
):
    features = torch.cat(feature_chunks, dim=0)
    image_ids = torch.cat(image_id_chunks, dim=0)
    random_keys = torch.cat(key_chunks, dim=0)

    if features.shape[0] > maximum_size:
        retained_indices = torch.topk(
            random_keys,
            k=maximum_size,
            largest=False,
        ).indices

        features = features[
            retained_indices
        ].contiguous()

        image_ids = image_ids[
            retained_indices
        ].contiguous()

        random_keys = random_keys[
            retained_indices
        ].contiguous()

    return (
        [features],
        [image_ids],
        [random_keys],
        int(features.shape[0]),
    )


@torch.no_grad()
def extract_candidate_pool(
    model,
    paths,
    category_seed,
):
    set_seed(category_seed)

    loader = make_image_loader(paths)

    cpu_generator = torch.Generator(
        device="cpu"
    )
    cpu_generator.manual_seed(
        category_seed + 777
    )

    feature_chunks = []
    image_id_chunks = []
    key_chunks = []

    retained_patch_count = 0
    total_sampled_patches = 0

    start = time.perf_counter()

    for images, image_indices, _ in tqdm(
        loader,
        desc="Candidate extraction",
    ):
        images_gpu = images.to(
            DEVICE,
            dtype=torch.float32,
            non_blocking=True,
        )

        features = model(images_gpu)

        batch_size, patch_count, _ = (
            features.shape
        )

        batch_features_list = []
        batch_image_ids_list = []

        for batch_index in range(batch_size):
            sample_count = min(
                PATCHES_PER_IMAGE,
                patch_count,
            )

            sampled_patch_indices = (
                torch.randperm(
                    patch_count,
                    device=DEVICE,
                )[:sample_count]
            )

            sampled_features = (
                features[
                    batch_index,
                    sampled_patch_indices,
                ]
                .detach()
                .float()
                .cpu()
            )

            source_image_id = int(
                image_indices[batch_index]
            )

            batch_features_list.append(
                sampled_features
            )

            batch_image_ids_list.append(
                torch.full(
                    (sample_count,),
                    source_image_id,
                    dtype=torch.long,
                )
            )

        batch_features = torch.cat(
            batch_features_list,
            dim=0,
        )

        batch_image_ids = torch.cat(
            batch_image_ids_list,
            dim=0,
        )

        current_count = int(
            batch_features.shape[0]
        )

        random_keys = torch.rand(
            current_count,
            generator=cpu_generator,
            dtype=torch.float32,
        )

        feature_chunks.append(batch_features)
        image_id_chunks.append(batch_image_ids)
        key_chunks.append(random_keys)

        retained_patch_count += current_count
        total_sampled_patches += current_count

        if (
            retained_patch_count
            > PRE_POOL + STREAM_POOL_MARGIN
        ):
            (
                feature_chunks,
                image_id_chunks,
                key_chunks,
                retained_patch_count,
            ) = compact_candidate_pool(
                feature_chunks,
                image_id_chunks,
                key_chunks,
                PRE_POOL,
            )

        del images_gpu, features
        del batch_features_list
        del batch_image_ids_list
        del batch_features
        del batch_image_ids
        del random_keys

    (
        feature_chunks,
        image_id_chunks,
        key_chunks,
        retained_patch_count,
    ) = compact_candidate_pool(
        feature_chunks,
        image_id_chunks,
        key_chunks,
        PRE_POOL,
    )

    candidate_features = (
        feature_chunks[0]
        .float()
        .contiguous()
    )

    candidate_image_ids = (
        image_id_chunks[0]
        .long()
        .contiguous()
    )

    elapsed = (
        time.perf_counter() - start
    )

    return (
        candidate_features,
        candidate_image_ids,
        elapsed,
        total_sampled_patches,
    )


@torch.no_grad()
def greedy_coreset_gpu(
    features_cpu,
    maximum_samples,
    chunk_size=CORESET_CHUNK,
    seed=SEED,
):
    """
    Coreset selection uses FP16 working features for speed,
    but selected bank values are returned from original FP32
    candidate features.

    Therefore Q1/Q2/Q3 memory banks themselves are FP32.
    """

    random.seed(seed)

    total_features = int(
        features_cpu.shape[0]
    )

    if total_features <= maximum_samples:
        return (
            features_cpu
            .clone()
            .float()
            .contiguous()
        )

    features_gpu = (
        features_cpu
        .to(
            DEVICE,
            non_blocking=True,
        )
        .half()
        .contiguous()
    )

    selected_indices = torch.empty(
        maximum_samples,
        dtype=torch.long,
        device=DEVICE,
    )

    first_index = random.randint(
        0,
        total_features - 1,
    )

    selected_indices[0] = first_index

    center = features_gpu[
        first_index:first_index + 1
    ]

    minimum_distances = torch.empty(
        total_features,
        dtype=torch.float32,
        device=DEVICE,
    )

    for start_index in range(
        0,
        total_features,
        chunk_size,
    ):
        end_index = min(
            start_index + chunk_size,
            total_features,
        )

        feature_chunk = features_gpu[
            start_index:end_index
        ]

        squared_distance = (
            feature_chunk - center
        ).float().pow(2).sum(dim=1)

        minimum_distances[
            start_index:end_index
        ] = squared_distance

    for selected_count in tqdm(
        range(1, maximum_samples),
        desc=f"Greedy coreset {maximum_samples}",
    ):
        farthest_index = torch.argmax(
            minimum_distances
        )

        selected_indices[
            selected_count
        ] = farthest_index

        center = features_gpu[
            farthest_index:farthest_index + 1
        ]

        for start_index in range(
            0,
            total_features,
            chunk_size,
        ):
            end_index = min(
                start_index + chunk_size,
                total_features,
            )

            feature_chunk = features_gpu[
                start_index:end_index
            ]

            squared_distance = (
                feature_chunk - center
            ).float().pow(2).sum(dim=1)

            minimum_distances[
                start_index:end_index
            ] = torch.minimum(
                minimum_distances[
                    start_index:end_index
                ],
                squared_distance,
            )

    selected_indices_cpu = (
        selected_indices.cpu()
    )

    selected_features = (
        features_cpu[
            selected_indices_cpu
        ]
        .float()
        .contiguous()
    )

    del features_gpu
    del selected_indices
    del selected_indices_cpu
    del minimum_distances

    clear_memory()

    return selected_features


# ============================================================
# H. FP32 SCORE USED ONLY FOR SELF-FILTERING
# ============================================================
@torch.no_grad()
def fp32_image_score(
    patch_features_gpu,
    memory_bank_gpu,
):
    patches = patch_features_gpu.float()

    patch_norm = (
        patches.pow(2)
        .sum(dim=1, keepdim=True)
    )

    min_sq = torch.full(
        (patches.shape[0],),
        float("inf"),
        device=DEVICE,
        dtype=torch.float32,
    )

    for start in range(
        0,
        memory_bank_gpu.shape[0],
        NN_CHUNK,
    ):
        memory_chunk = (
            memory_bank_gpu[
                start:start + NN_CHUNK
            ]
            .float()
        )

        memory_norm = (
            memory_chunk.pow(2)
            .sum(dim=1)
            .unsqueeze(0)
        )

        squared_distance = (
            patch_norm
            + memory_norm
            - 2.0 * (
                patches
                @ memory_chunk.t()
            )
        ).clamp_min(0.0)

        min_sq = torch.minimum(
            min_sq,
            squared_distance
            .min(dim=1)
            .values,
        )

    return float(
        min_sq.sqrt().max().item()
    )


@torch.no_grad()
def score_training_images_for_filter(
    model,
    train_paths,
    q1_bank_cpu,
):
    loader = make_image_loader(
        train_paths
    )

    q1_bank_gpu = (
        q1_bank_cpu
        .to(
            DEVICE,
            dtype=torch.float32,
            non_blocking=True,
        )
        .contiguous()
    )

    scores = np.empty(
        len(train_paths),
        dtype=np.float32,
    )

    for images, image_indices, _ in tqdm(
        loader,
        desc="Self-filter scoring",
    ):
        images_gpu = images.to(
            DEVICE,
            dtype=torch.float32,
            non_blocking=True,
        )

        features = model(images_gpu)

        for i in range(
            features.shape[0]
        ):
            image_id = int(
                image_indices[i]
            )

            scores[image_id] = (
                fp32_image_score(
                    features[i],
                    q1_bank_gpu,
                )
            )

        del images_gpu, features

    del q1_bank_gpu
    clear_memory()

    return scores


# ============================================================
# I. PHASE 1 — BUILD Q1/Q2/Q3 BANKS ONCE PER CATEGORY
# ============================================================
all_ranking_frames = []
bank_metadata_rows = []

print(
    "\n"
    + "=" * 110
    + "\nPHASE 1: BUILD FILTER BANK + Q3 10%-FILTERED 5k BANK\n"
    + "=" * 110
)

for category_index, category in enumerate(
    categories,
    start=1,
):
    print("\n" + "#" * 110)
    print(
        f"CATEGORY "
        f"{category_index}/"
        f"{len(categories)}: "
        f"{category}"
    )
    print("#" * 110)

    category_artifact_dir = LOCAL_ARTIFACT_DIR / category
    q3_checkpoint = category_artifact_dir / "Q3_bank_fp32.pt"
    clean_checkpoint = category_artifact_dir / "clean_train_paths.csv"

    # Resume completed Phase-1 work from Drive.
    if q3_checkpoint.exists() and clean_checkpoint.exists():
        print("Phase-1 checkpoint found in Drive -> reusing existing Q3 bank.")
        clean_frame = pd.read_csv(clean_checkpoint)
        bank_metadata_rows.append({
            "Category": category,
            "Train_Images": len(list_images(DATASET_ROOT / category / "train" / "good")),
            "Filtered_Images": np.nan,
            "Clean_Train_Images": len(clean_frame),
            "Candidate_Patches": np.nan,
            "Clean_Candidate_Patches": np.nan,
            "Filter_Bank_Size": np.nan,
            "Auxiliary_5k_Bank_Size": np.nan,
            "Q3_Bank_Size": int(torch.load(q3_checkpoint, map_location="cpu").shape[0]),
            "Candidate_Extraction_sec": 0.0,
            "Total_Sampled_Patches": np.nan,
            "Resumed_From_Checkpoint": True,
        })
        continue

    set_seed(SEED)

    category_root = (
        DATASET_ROOT / category
    )

    train_paths = list_images(
        category_root
        / "train"
        / "good"
    )

    if len(train_paths) < 2:
        raise RuntimeError(
            f"{category}: "
            "at least two train-good "
            "images are required."
        )

    (
        candidate_pool,
        candidate_image_ids,
        extraction_time_sec,
        total_sampled_patches,
    ) = extract_candidate_pool(
        training_extractor,
        train_paths,
        SEED,
    )

    print(
        "Candidate pool:",
        tuple(candidate_pool.shape),
    )

    # ----------------------------------------
    # 20k filter bank: 0% filter, 20k, FP32
    # ----------------------------------------
    print("\nBuilding 20k filter-scoring bank...")

    q1_bank = greedy_coreset_gpu(
        candidate_pool,
        min(
            FILTER_SCORING_BANK_SIZE,
            candidate_pool.shape[0],
        ),
        CORESET_CHUNK,
        SEED,
    )

    # ----------------------------------------
    # auxiliary 5k bank: 0% filter, 5k, FP32
    # ----------------------------------------
    print("\nBuilding auxiliary unfiltered 5k bank (not evaluated)...")

    q2_bank = greedy_coreset_gpu(
        candidate_pool,
        min(
            FINAL_MEMORY_SIZE,
            candidate_pool.shape[0],
        ),
        CORESET_CHUNK,
        SEED,
    )

    # ----------------------------------------
    # Rank train-normal images with 20k filter bank.
    # ----------------------------------------
    print(
        "\nRanking training-normal "
        "images for 10% self-filter..."
    )

    train_scores = (
        score_training_images_for_filter(
            training_extractor,
            train_paths,
            q1_bank,
        )
    )

    ranking_frame = pd.DataFrame({
        "category": category,
        "train_image_id": np.arange(
            len(train_paths),
            dtype=np.int32,
        ),
        "image_path": [
            str(p) for p in train_paths
        ],
        "self_filter_score": train_scores,
    }).sort_values(
        "self_filter_score",
        ascending=False,
    ).reset_index(drop=True)

    ranking_frame[
        "suspicion_rank"
    ] = np.arange(
        1,
        len(ranking_frame) + 1,
        dtype=np.int32,
    )

    remove_count = int(
        round(
            len(train_paths)
            * FILTER_PERCENT
            / 100.0
        )
    )

    remove_count = max(
        1,
        min(
            remove_count,
            len(train_paths) - 1,
        ),
    )

    removed_image_ids = set(
        ranking_frame
        .iloc[:remove_count][
            "train_image_id"
        ]
        .astype(int)
        .tolist()
    )

    ranking_frame[
        "Filter_Percent"
    ] = FILTER_PERCENT

    ranking_frame[
        "filtered_out"
    ] = (
        ranking_frame[
            "train_image_id"
        ]
        .isin(removed_image_ids)
    )

    all_ranking_frames.append(
        ranking_frame
    )

    removed_mask = torch.zeros(
        len(train_paths),
        dtype=torch.bool,
    )

    removed_mask[
        list(removed_image_ids)
    ] = True

    keep_patch_mask = (
        ~removed_mask[
            candidate_image_ids
        ]
    )

    clean_candidate_pool = (
        candidate_pool[
            keep_patch_mask
        ]
        .contiguous()
    )

    clean_train_paths = [
        path
        for image_id, path
        in enumerate(train_paths)
        if image_id
        not in removed_image_ids
    ]

    print(
        f"Filtered images: "
        f"{len(removed_image_ids)}"
        f"/{len(train_paths)}"
    )

    print(
        "Candidate patches:",
        int(candidate_pool.shape[0]),
        "->",
        int(clean_candidate_pool.shape[0]),
    )

    # ----------------------------------------
    # Q3 bank: 10% filter, 5k, FP32
    # ----------------------------------------
    print("\nBuilding Q3 10%-filtered 5k bank...")

    q3_bank = greedy_coreset_gpu(
        clean_candidate_pool,
        min(
            FINAL_MEMORY_SIZE,
            clean_candidate_pool.shape[0],
        ),
        CORESET_CHUNK,
        SEED,
    )

    # ----------------------------------------
    # Save local banks.
    # Q4/Q5 will reuse EXACT Q3 bank values,
    # changing only storage/scoring precision.
    # ----------------------------------------
    category_artifact_dir = (
        LOCAL_ARTIFACT_DIR / category
    )

    category_artifact_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        q1_bank,
        category_artifact_dir
        / "Q1_bank_fp32.pt",
    )

    torch.save(
        q2_bank,
        category_artifact_dir
        / "Q2_bank_fp32.pt",
    )

    torch.save(
        q3_bank,
        category_artifact_dir
        / "Q3_bank_fp32.pt",
    )

    pd.DataFrame({
        "clean_train_path": [
            str(p)
            for p in clean_train_paths
        ]
    }).to_csv(
        category_artifact_dir
        / "clean_train_paths.csv",
        index=False,
    )

    bank_metadata_rows.append({
        "Category": category,
        "Train_Images": len(train_paths),
        "Filtered_Images": len(
            removed_image_ids
        ),
        "Clean_Train_Images": len(
            clean_train_paths
        ),
        "Candidate_Patches": int(
            candidate_pool.shape[0]
        ),
        "Clean_Candidate_Patches": int(
            clean_candidate_pool.shape[0]
        ),
        "Filter_Bank_Size": int(
            q1_bank.shape[0]
        ),
        "Auxiliary_5k_Bank_Size": int(
            q2_bank.shape[0]
        ),
        "Q3_Bank_Size": int(
            q3_bank.shape[0]
        ),
        "Candidate_Extraction_sec": (
            extraction_time_sec
        ),
        "Total_Sampled_Patches": int(
            total_sampled_patches
        ),
        "Resumed_From_Checkpoint": False,
    })

    del candidate_pool
    del candidate_image_ids
    del clean_candidate_pool
    del q1_bank, q2_bank, q3_bank
    del train_scores

    clear_memory()


if all_ranking_frames:
    df_rankings = pd.concat(
        all_ranking_frames,
        ignore_index=True,
    )
    df_rankings.to_csv(
        RANKINGS_CSV,
        index=False,
    )
elif RANKINGS_CSV.exists():
    df_rankings = pd.read_csv(RANKINGS_CSV)
    print("Reusing existing self-filter rankings:", RANKINGS_CSV)
else:
    df_rankings = pd.DataFrame()

df_bank_metadata = pd.DataFrame(
    bank_metadata_rows
)

print("\nPHASE 1 COMPLETE")
display(df_bank_metadata)


# Free training extractor before deployment measurements.
del training_extractor
clear_memory()


# ============================================================
# J. INT8 CALIBRATION SET — TRAIN NORMAL ONLY
# ============================================================
# Use the Q3-clean training-normal paths across categories.
all_clean_training_paths = []

for category in categories:
    clean_csv = (
        LOCAL_ARTIFACT_DIR
        / category
        / "clean_train_paths.csv"
    )

    clean_frame = pd.read_csv(
        clean_csv
    )

    all_clean_training_paths.extend(
        [
            Path(p)
            for p
            in clean_frame[
                "clean_train_path"
            ].tolist()
        ]
    )

rng = np.random.default_rng(
    SEED + 9000
)

calibration_count = min(
    INT8_CALIBRATION_IMAGES,
    len(all_clean_training_paths),
)

calibration_indices = rng.permutation(
    len(all_clean_training_paths)
)[:calibration_count]

calibration_paths = [
    all_clean_training_paths[i]
    for i in calibration_indices
]

calibration_loader = make_image_loader(
    calibration_paths,
    batch_size=INT8_CALIBRATION_BATCH,
)

print(
    "\nINT8 calibration images:",
    len(calibration_paths),
)
print(
    "Calibration data: "
    "10%-filtered Lusitano training-normal only"
)


# ============================================================
# K. PRECISION-AWARE PATCHCORE NN SCORER
# ============================================================
@torch.no_grad()
def anomaly_score(
    patch_features_gpu,
    memory_bank_gpu,
    nn_dtype,
):
    """
    Q1/Q2/Q3:
        FP32 feature-bank matmul.

    Q4/Q5:
        FP16 feature-bank matmul.

    For FP16, squared norms and final distance
    arithmetic are accumulated in FP32.
    """

    patches = patch_features_gpu.to(
        dtype=nn_dtype
    )

    patch_norm = (
        patches.float()
        .pow(2)
        .sum(dim=1, keepdim=True)
    )

    minimum_squared_distance = torch.full(
        (patches.shape[0],),
        float("inf"),
        device=DEVICE,
        dtype=torch.float32,
    )

    for start_index in range(
        0,
        memory_bank_gpu.shape[0],
        NN_CHUNK,
    ):
        memory_chunk = (
            memory_bank_gpu[
                start_index:
                start_index + NN_CHUNK
            ]
            .to(dtype=nn_dtype)
        )

        memory_norm = (
            memory_chunk.float()
            .pow(2)
            .sum(dim=1)
            .unsqueeze(0)
        )

        dot_product = (
            patches
            @ memory_chunk.t()
        ).float()

        squared_distance = (
            patch_norm
            + memory_norm
            - 2.0 * dot_product
        ).clamp_min(0.0)

        minimum_squared_distance = (
            torch.minimum(
                minimum_squared_distance,
                squared_distance
                .min(dim=1)
                .values,
            )
        )

    return float(
        minimum_squared_distance
        .sqrt()
        .max()
        .item()
    )


@torch.no_grad()
def model_forward(
    model,
    images_cpu,
    input_dtype,
):
    images_gpu = images_cpu.to(
        DEVICE,
        dtype=input_dtype,
        non_blocking=True,
    )

    features = model(images_gpu)

    return images_gpu, features


# ============================================================
# L. THRESHOLD + TEST EVALUATION
# ============================================================
@torch.no_grad()
def compute_threshold(
    model,
    threshold_paths,
    memory_bank_gpu,
    input_dtype,
    nn_dtype,
    seed,
):
    rng = np.random.default_rng(
        seed + 100
    )

    subset_count = min(
        THRESH_SAMPLE_IMAGES,
        len(threshold_paths),
    )

    selected_indices = rng.permutation(
        len(threshold_paths)
    )[:subset_count]

    paths = [
        threshold_paths[i]
        for i in selected_indices
    ]

    loader = make_image_loader(
        paths
    )

    threshold_scores = []

    start = time.perf_counter()

    for images, _, _ in tqdm(
        loader,
        desc="Threshold",
    ):
        images_gpu, features = (
            model_forward(
                model,
                images,
                input_dtype,
            )
        )

        for batch_index in range(
            features.shape[0]
        ):
            threshold_scores.append(
                anomaly_score(
                    features[
                        batch_index
                    ],
                    memory_bank_gpu,
                    nn_dtype,
                )
            )

        del images_gpu, features

    torch.cuda.synchronize()

    elapsed = (
        time.perf_counter() - start
    )

    threshold_scores = np.asarray(
        threshold_scores,
        dtype=np.float32,
    )

    ddof = (
        1
        if len(threshold_scores) > 1
        else 0
    )

    threshold = float(
        threshold_scores.mean()
        + 3.0
        * threshold_scores.std(
            ddof=ddof
        )
    )

    return (
        threshold,
        threshold_scores,
        elapsed,
    )


@torch.no_grad()
def warm_up(
    model,
    test_loader,
    memory_bank_gpu,
    input_dtype,
    nn_dtype,
):
    first_batch = next(
        iter(test_loader)
    )

    images = first_batch[0][:1]

    for _ in range(
        WARMUP_BATCHES
    ):
        images_gpu, features = (
            model_forward(
                model,
                images,
                input_dtype,
            )
        )

        _ = anomaly_score(
            features[0],
            memory_bank_gpu,
            nn_dtype,
        )

    torch.cuda.synchronize()

    del images_gpu, features
    clear_memory()


@torch.no_grad()
def evaluate_test_set(
    model,
    memory_bank_gpu,
    threshold,
    test_loader,
    input_dtype,
    nn_dtype,
):
    warm_up(
        model,
        test_loader,
        memory_bank_gpu,
        input_dtype,
        nn_dtype,
    )

    reset_peak_gpu_memory()

    true_labels = []
    anomaly_scores = []
    image_paths = []

    compute_total_sec = 0.0

    end_to_end_start = (
        time.perf_counter()
    )

    for images, labels, paths in tqdm(
        test_loader,
        desc="Testing",
    ):
        torch.cuda.synchronize()

        compute_start = (
            time.perf_counter()
        )

        images_gpu, features = (
            model_forward(
                model,
                images,
                input_dtype,
            )
        )

        batch_scores = [
            anomaly_score(
                features[batch_index],
                memory_bank_gpu,
                nn_dtype,
            )
            for batch_index
            in range(features.shape[0])
        ]

        torch.cuda.synchronize()

        compute_total_sec += (
            time.perf_counter()
            - compute_start
        )

        anomaly_scores.extend(
            batch_scores
        )

        true_labels.extend(
            int(label)
            for label in labels
        )

        image_paths.extend(
            str(path)
            for path in paths
        )

        del images_gpu
        del features
        del batch_scores

    torch.cuda.synchronize()

    e2e_total_sec = (
        time.perf_counter()
        - end_to_end_start
    )

    y_true = np.asarray(
        true_labels,
        dtype=np.int32,
    )

    scores = np.asarray(
        anomaly_scores,
        dtype=np.float32,
    )

    y_pred = (
        scores > threshold
    ).astype(np.int32)

    auc_roc = (
        float(
            roc_auc_score(
                y_true,
                scores,
            )
        )
        if len(
            np.unique(y_true)
        ) == 2
        else np.nan
    )

    average_precision = (
        float(
            average_precision_score(
                y_true,
                scores,
            )
        )
        if np.any(y_true == 1)
        else np.nan
    )

    f1 = float(
        f1_score(
            y_true,
            y_pred,
            zero_division=0,
        )
    )

    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        )
        .ravel()
    )

    image_count = max(
        len(y_true),
        1,
    )

    compute_ms = (
        compute_total_sec
        / image_count
        * 1000.0
    )

    e2e_ms = (
        e2e_total_sec
        / image_count
        * 1000.0
    )

    throughput_fps = (
        1000.0 / e2e_ms
        if e2e_ms > 0
        else 0.0
    )

    return {
        "AUC_ROC": auc_roc,
        "mAP_AP": average_precision,
        "F1_Score": f1,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Compute_Latency_ms_per_image": (
            float(compute_ms)
        ),
        "End_To_End_Latency_ms_per_image": (
            float(e2e_ms)
        ),
        "Throughput_FPS": float(
            throughput_fps
        ),
        "Peak_GPU_Memory_MB": float(
            peak_gpu_memory_mb()
        ),
    }


# ============================================================
# M. CONFIGURATION TABLE
# ============================================================
CONFIG_TABLE = pd.DataFrame([
    {"ID": "Q3", "Filter": "10%", "Bank": 5000, "Backbone": "FP32", "Bank_precision": "FP32"},
    {"ID": "Q4", "Filter": "10%", "Bank": 5000, "Backbone": "FP16", "Bank_precision": "FP16"},
    {"ID": "Q5", "Filter": "10%", "Bank": 5000, "Backbone": "INT8", "Bank_precision": "FP16"},
])
print("\nLUSITANO EXTERNAL VALIDATION TABLE")
display(CONFIG_TABLE)


# N. COMMON EVALUATION FUNCTION FOR ONE CONFIGURATION
# ============================================================
# Resume results ONLY when they were generated under this exact runtime.
# The Phase-1 5k Q3 bank is algorithmic and is always safe to reuse, but
# deployment latency/FPS must not be mixed across TensorRT/Torch-TensorRT versions.
CURRENT_RUNTIME = {
    "Runtime_PyTorch": str(torch.__version__),
    "Runtime_Torch_TensorRT": str(getattr(torch_tensorrt, "__version__", "unknown")),
    "Runtime_TensorRT": str(trt.__version__),
    "Runtime_GPU": str(torch.cuda.get_device_name(0)),
}

all_result_rows = []
if RESULTS_CSV.exists():
    try:
        _existing_results = pd.read_csv(RESULTS_CSV)
        _runtime_cols = list(CURRENT_RUNTIME.keys())
        _runtime_cols_present = all(c in _existing_results.columns for c in _runtime_cols)
        _runtime_matches = False
        if _runtime_cols_present and len(_existing_results):
            _runtime_matches = all(
                _existing_results[c].astype(str).eq(str(v)).all()
                for c, v in CURRENT_RUNTIME.items()
            )
        if _runtime_matches:
            _existing_results = _existing_results[
                _existing_results["ID"].astype(str).isin(["Q3", "Q4", "Q5"])
            ].copy()
            all_result_rows = _existing_results.to_dict("records")
            if all_result_rows:
                print(
                    "Resuming same-runtime configuration results:",
                    sorted({str(r.get("ID")) for r in all_result_rows}),
                )
        else:
            print(
                "Existing deployment results use a different/unrecorded runtime -> "
                "Q3/Q4/Q5 inference will be rerun fairly under the current stack. "
                "The saved Phase-1 5k memory bank will still be reused."
            )
    except Exception as _resume_error:
        print("Existing results could not be resumed; starting result table fresh:", _resume_error)


def evaluate_configuration_across_itd(
    config_id,
    model,
    input_dtype,
    nn_dtype,
    bank_dtype,
    backbone_precision,
    bank_precision,
    filter_percent,
    bank_filename,
    threshold_uses_clean_train,
    backbone_estimated_mb,
):
    global all_result_rows

    print(
        "\n"
        + "=" * 110
    )
    print(
        f"EVALUATING {config_id}: "
        f"backbone={backbone_precision}, "
        f"bank={bank_precision}, "
        f"filter={filter_percent}%, "
        f"bank file={bank_filename}"
    )
    print(
        "=" * 110
    )

    for category_index, category in enumerate(
        categories,
        start=1,
    ):
        if any(
            str(r.get("ID")) == str(config_id)
            and str(r.get("Category")) == str(category)
            for r in all_result_rows
        ):
            print(f"{config_id} | {category}: saved result found -> SKIP")
            continue

        print("\n" + "#" * 100)
        print(
            f"{config_id} | CATEGORY "
            f"{category_index}/"
            f"{len(categories)}: "
            f"{category}"
        )
        print("#" * 100)

        category_root = (
            DATASET_ROOT / category
        )

        train_paths = list_images(
            category_root
            / "train"
            / "good"
        )

        if threshold_uses_clean_train:
            clean_frame = pd.read_csv(
                LOCAL_ARTIFACT_DIR
                / category
                / "clean_train_paths.csv"
            )

            threshold_paths = [
                Path(p)
                for p
                in clean_frame[
                    "clean_train_path"
                ].tolist()
            ]
        else:
            threshold_paths = train_paths

        test_good_paths = list_images(
            category_root
            / "test"
            / "good"
        )

        test_anomaly_paths = list_images(
            category_root
            / "test"
            / "anomaly"
        )

        if (
            not test_good_paths
            or not test_anomaly_paths
        ):
            raise RuntimeError(
                f"{category}: "
                "test/good and test/anomaly "
                "are both required."
            )

        test_items = (
            [
                (p, 0)
                for p in test_good_paths
            ]
            + [
                (p, 1)
                for p
                in test_anomaly_paths
            ]
        )

        test_loader = make_test_loader(
            test_items
        )

        bank_cpu = torch.load(
            LOCAL_ARTIFACT_DIR
            / category
            / bank_filename,
            map_location="cpu",
        )

        bank_cpu = (
            bank_cpu
            .to(dtype=bank_dtype)
            .contiguous()
        )

        memory_bank_size_mb = (
            tensor_size_mb(
                bank_cpu
            )
        )

        memory_bank_gpu = (
            bank_cpu
            .to(
                DEVICE,
                dtype=bank_dtype,
                non_blocking=True,
            )
            .contiguous()
        )

        (
            threshold,
            threshold_scores,
            threshold_time_sec,
        ) = compute_threshold(
            model,
            threshold_paths,
            memory_bank_gpu,
            input_dtype,
            nn_dtype,
            SEED,
        )

        metrics = evaluate_test_set(
            model,
            memory_bank_gpu,
            threshold,
            test_loader,
            input_dtype,
            nn_dtype,
        )

        total_footprint_mb = (
            backbone_estimated_mb
            + memory_bank_size_mb
        )

        row = {
            "ID": config_id,
            **CURRENT_RUNTIME,
            "Category": category,
            "Dataset": "Lusitano",
            "Seed": SEED,
            "Filter_Percent": (
                filter_percent
            ),
            "Bank_Size": int(
                bank_cpu.shape[0]
            ),
            "Backbone": (
                "EfficientNet-B5 "
                "features[7]"
            ),
            "Backbone_Precision": (
                backbone_precision
            ),
            "Bank_Precision": (
                bank_precision
            ),
            "NN_Matmul_Precision": (
                "FP32"
                if nn_dtype
                == torch.float32
                else (
                    "FP16; norms/final "
                    "distance accumulated "
                    "in FP32"
                )
            ),
            "IMG_SIZE": IMG_SIZE,
            "Feature_Dimension": (
                FEATURE_DIMENSION
            ),
            "Patch_Grid_Count": (
                PATCH_GRID_COUNT
            ),
            "Patches_Per_Image": (
                PATCHES_PER_IMAGE
            ),
            "Threshold_Method": (
                "train-normal "
                "mean + 3*std"
            ),
            "Threshold_Train_Set": (
                "10%-filtered train-good"
                if threshold_uses_clean_train
                else "all train-good"
            ),
            "Threshold": float(
                threshold
            ),
            "Threshold_Time_sec": float(
                threshold_time_sec
            ),
            "Threshold_Images": int(
                len(threshold_scores)
            ),
            "Test_Good_Images": int(
                len(test_good_paths)
            ),
            "Test_Anomaly_Images": int(
                len(test_anomaly_paths)
            ),
            **metrics,
            "Memory_Bank_Size_MB": (
                memory_bank_size_mb
            ),
            "Estimated_Backbone_Weight_MB": (
                backbone_estimated_mb
            ),
            "Estimated_Total_Footprint_MB": (
                total_footprint_mb
            ),
        }

        all_result_rows.append(
            row
        )

        pd.DataFrame(
            all_result_rows
        ).to_csv(
            RESULTS_CSV,
            index=False,
        )

        print(
            f"AUC-ROC : "
            f"{metrics['AUC_ROC']:.6f}"
        )
        print(
            f"AP      : "
            f"{metrics['mAP_AP']:.6f}"
        )
        print(
            f"F1      : "
            f"{metrics['F1_Score']:.6f}"
        )
        print(
            f"Compute : "
            f"{metrics['Compute_Latency_ms_per_image']:.3f} "
            "ms/image"
        )
        print(
            f"E2E     : "
            f"{metrics['End_To_End_Latency_ms_per_image']:.3f} "
            "ms/image"
        )
        print(
            f"FPS     : "
            f"{metrics['Throughput_FPS']:.3f}"
        )
        print(
            f"Footprint est.: "
            f"{total_footprint_mb:.3f} MB"
        )

        del bank_cpu
        del memory_bank_gpu
        del threshold_scores

        clear_memory()


# ============================================================
# O. COMPILE STRICT FP32 TENSORRT BACKBONE
# ============================================================
# Final PyTorch 2.11 compatibility check before torch.export/TensorRT.
print('cuDNN legacy TF32 getter before Q3:', torch.backends.cudnn.allow_tf32)
print(
    "\n"
    + "=" * 110
    + "\nPHASE 2: FP32 TENSORRT — Q3 EXTERNAL VALIDATION\n"
    + "=" * 110
)

fp32_source = (
    EfficientNetB5Layer7()
    .to(DEVICE)
    .eval()
)

fp32_input_spec = torch_tensorrt.Input(
    min_shape=(
        1,
        3,
        IMG_SIZE,
        IMG_SIZE,
    ),
    opt_shape=(
        BATCH_SIZE,
        3,
        IMG_SIZE,
        IMG_SIZE,
    ),
    max_shape=(
        BATCH_SIZE,
        3,
        IMG_SIZE,
        IMG_SIZE,
    ),
    dtype=torch.float32,
)

fp32_trt_model = (
    torch_tensorrt.compile(
        fp32_source,
        ir="dynamo",
        inputs=[fp32_input_spec],
        # Dynamo strong typing respects the FP32 model/input dtypes.
        # Do not combine explicit typing with deprecated enabled_precisions.
        min_block_size=1,
        use_explicit_typing=True,
        disable_tf32=True,
    )
)

# Trigger engine build.
with torch.no_grad():
    _ = fp32_trt_model(
        torch.zeros(
            1,
            3,
            IMG_SIZE,
            IMG_SIZE,
            device=DEVICE,
            dtype=torch.float32,
        )
    )

torch.cuda.synchronize()

del fp32_source
clear_memory()


# Q3
evaluate_configuration_across_itd(
    config_id="Q3",
    model=fp32_trt_model,
    input_dtype=torch.float32,
    nn_dtype=torch.float32,
    bank_dtype=torch.float32,
    backbone_precision="FP32 TensorRT",
    bank_precision="FP32",
    filter_percent=FILTER_PERCENT,
    bank_filename="Q3_bank_fp32.pt",
    threshold_uses_clean_train=True,
    backbone_estimated_mb=(
        FP32_BACKBONE_EST_MB
    ),
)


del fp32_trt_model
clear_memory()


# ============================================================
# P. COMPILE FP16 TENSORRT BACKBONE — Q4
# ============================================================
print(
    "\n"
    + "=" * 110
    + "\nPHASE 3: FP16 TENSORRT — Q4\n"
    + "=" * 110
)

fp16_source = (
    EfficientNetB5Layer7()
    .half()
    .to(DEVICE)
    .eval()
)

fp16_input_spec = torch_tensorrt.Input(
    min_shape=(
        1,
        3,
        IMG_SIZE,
        IMG_SIZE,
    ),
    opt_shape=(
        BATCH_SIZE,
        3,
        IMG_SIZE,
        IMG_SIZE,
    ),
    max_shape=(
        BATCH_SIZE,
        3,
        IMG_SIZE,
        IMG_SIZE,
    ),
    dtype=torch.float16,
)

print("Q4 compile: FP16 model + FP16 input under TensorRT explicit typing")

fp16_trt_model = (
    torch_tensorrt.compile(
        fp16_source,
        ir="dynamo",
        inputs=[fp16_input_spec],
        # Torch-TensorRT Dynamo explicit typing controls FP16 from model/input dtypes.
        # Do NOT combine enabled_precisions={FP16} with use_explicit_typing=True.
        min_block_size=1,
        use_explicit_typing=True,
        disable_tf32=True,
    )
)

with torch.no_grad():
    _ = fp16_trt_model(
        torch.zeros(
            1,
            3,
            IMG_SIZE,
            IMG_SIZE,
            device=DEVICE,
            dtype=torch.float16,
        )
    )

torch.cuda.synchronize()

del fp16_source
clear_memory()


evaluate_configuration_across_itd(
    config_id="Q4",
    model=fp16_trt_model,
    input_dtype=torch.float16,
    nn_dtype=torch.float16,
    bank_dtype=torch.float16,
    backbone_precision="FP16 TensorRT",
    bank_precision="FP16",
    filter_percent=FILTER_PERCENT,
    bank_filename="Q3_bank_fp32.pt",
    threshold_uses_clean_train=True,
    backbone_estimated_mb=(
        FP16_BACKBONE_EST_MB
    ),
)


del fp16_trt_model
clear_memory()


# ============================================================
# Q. INT8 MODELOPT PTQ + TENSORRT — Q5
# ============================================================
print("\n" + "=" * 110 + "\nPHASE 4: INT8 MODELOPT PTQ + TENSORRT — Q5\n" + "=" * 110)

# Import ModelOpt only now. Q3/Q4 and the Q3 bank are already saved.
import modelopt.torch.quantization as mtq

int8_source = EfficientNetB5Layer7().to(DEVICE).eval()
for parameter in int8_source.parameters():
    parameter.requires_grad = False

@torch.no_grad()
def int8_calibration_loop(model):
    model.eval()
    for images, _, _ in tqdm(calibration_loader, desc="INT8 calibration"):
        images_gpu = images.to(DEVICE, dtype=torch.float32, non_blocking=True)
        _ = model(images_gpu)
        del images_gpu

print("Applying ModelOpt INT8 PTQ...")
mtq.quantize(int8_source, mtq.INT8_DEFAULT_CFG, forward_loop=int8_calibration_loop)
print("INT8 calibration complete.")

from modelopt.torch.quantization.utils import export_torch_mode
batch_dim = torch.export.Dim("batch", min=1, max=BATCH_SIZE)
example_input = torch.randn(
    BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE,
    device=DEVICE, dtype=torch.float32,
)

print("Exporting ModelOpt INT8 graph...")
with export_torch_mode():
    int8_exported_program = torch.export.export(
        int8_source,
        (example_input,),
        dynamic_shapes=({0: batch_dim},),
        strict=False,
    )
print("ModelOpt INT8 torch.export: SUCCESS")

int8_input_spec = torch_tensorrt.Input(
    min_shape=(1, 3, IMG_SIZE, IMG_SIZE),
    opt_shape=(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE),
    max_shape=(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE),
    dtype=torch.float32,
)

print("Compiling exported INT8 model with TensorRT...")
int8_trt_model = torch_tensorrt.dynamo.compile(
    int8_exported_program,
    arg_inputs=[int8_input_spec],
    min_block_size=1,
)

with torch.no_grad():
    _test = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE, dtype=torch.float32)
    _out = int8_trt_model(_test)
    print("INT8 TensorRT forward pass: SUCCESS", tuple(_out.shape), _out.dtype)
    del _test, _out
torch.cuda.synchronize()

del int8_source, int8_exported_program, example_input
clear_memory()

evaluate_configuration_across_itd(
    config_id="Q5",
    model=int8_trt_model,
    input_dtype=torch.float32,
    nn_dtype=torch.float16,
    bank_dtype=torch.float16,
    backbone_precision="INT8 ModelOpt PTQ + TensorRT",
    bank_precision="FP16",
    filter_percent=FILTER_PERCENT,
    bank_filename="Q3_bank_fp32.pt",
    threshold_uses_clean_train=True,
    backbone_estimated_mb=INT8_BACKBONE_EST_MB,
)

del int8_trt_model
clear_memory()


# R. FINAL Q3-Q5 EXTERNAL VALIDATION TABLES
# ============================================================
df_results = pd.DataFrame(all_result_rows)
df_results.to_csv(RESULTS_CSV, index=False)

summary_cols = [
    "ID", "Filter_Percent", "Bank_Size", "Backbone_Precision", "Bank_Precision",
    "AUC_ROC", "mAP_AP", "F1_Score",
    "Compute_Latency_ms_per_image", "End_To_End_Latency_ms_per_image",
    "Throughput_FPS", "Estimated_Total_Footprint_MB", "Peak_GPU_Memory_MB",
]
summary = df_results[summary_cols].copy().sort_values("ID").reset_index(drop=True)

comparisons = [("Q3", "Q4", "FP32 -> FP16"), ("Q4", "Q5", "FP16 -> INT8 backbone")]
metric_cols = [
    "AUC_ROC", "mAP_AP", "F1_Score",
    "Compute_Latency_ms_per_image", "End_To_End_Latency_ms_per_image",
    "Throughput_FPS", "Estimated_Total_Footprint_MB", "Peak_GPU_Memory_MB",
]
indexed = summary.set_index("ID")
delta_rows = []
for before_id, after_id, question in comparisons:
    row = {"Comparison": f"{before_id}->{after_id}", "Question": question}
    for metric in metric_cols:
        before = float(indexed.loc[before_id, metric])
        after = float(indexed.loc[after_id, metric])
        change = after - before
        pct = 100.0 * change / abs(before) if before != 0 else np.nan
        row[f"{metric}_Before"] = before
        row[f"{metric}_After"] = after
        row[f"{metric}_Absolute_Change"] = change
        row[f"{metric}_Percent_Change"] = pct
    delta_rows.append(row)

delta_table = pd.DataFrame(delta_rows)
delta_table.to_csv(DELTAS_CSV, index=False)

run_info = pd.DataFrame([{
    "Dataset": "Lusitano",
    **CURRENT_RUNTIME,
    "Seed": SEED,
    "Train_Normal_Images": len(list_images(DATASET_ROOT / "lusitano" / "train" / "good")),
    "Test_Normal_Images": len(list_images(DATASET_ROOT / "lusitano" / "test" / "good")),
    "Test_Defect_Images": len(list_images(DATASET_ROOT / "lusitano" / "test" / "anomaly")),
    "Filter_Percent": FILTER_PERCENT,
    "Final_Memory_Bank": FINAL_MEMORY_SIZE,
    "Backbone": "EfficientNet-B5 features[7]",
    "Image_Size": IMG_SIZE,
    "Calibration_Images": len(calibration_paths),
    "Dataset_Copy_Time_sec": DATASET_COPY_TIME_SEC,
}])
run_info.to_csv(RUN_INFO_CSV, index=False)

print("\n" + "=" * 110)
print("LUSITANO Q3-Q5 EXTERNAL DEPLOYMENT VALIDATION")
print("=" * 110)
display(summary)

print("\nPAIRWISE DEPLOYMENT CHANGES")
display(delta_table)

print("\nSaved:")
print("Results :", RESULTS_CSV)
print("Deltas  :", DELTAS_CSV)
print("Ranking :", RANKINGS_CSV)
print("Run info:", RUN_INFO_CSV)
print("\nLusitano Q3-Q5 validation COMPLETE.")

